Exercise 1: KV-Cache for Decoder-Only Transformers



[← Back to reading list]()

# Exercise 1: KV-Cache for Decoder-Only Transformers

> **Learning objectives:**

**Learning objectives:**


- Understand why autoregressive generation is slow without caching

- Implement a KV-Cache that stores key-value pairs from previous tokens

- Modify a transformer's attention to use cached K,V tensors

- Achieve significant speedups for autoregressive generation

**Prerequisites:** A working decoder-only transformer implementation (e.g. from [ARENA Chapter 1.1]()). You should have implemented `Attention`, `MLP`, `TransformerBlock`, and a full `Transformer` class.

## Setup

Start by copying your working transformer code from ARENA into a new notebook. You'll also need these imports:

In [ ]:
import torch
import torch as t
from torch import nn
import time
from dataclasses import dataclass

device = "cuda" if torch.cuda.is_available() else "cpu"

## Background: The Problem with Naive Generation

In autoregressive generation, we produce tokens one at a time. At each step, we feed the *entire* sequence so far through the model to get the next token prediction. This means:


- Step 1: process 1 token

- Step 2: process 2 tokens

- Step \(n\): process \(n\) tokens

The total work is \(O(n^2)\) in the sequence length. But notice: at step \(n\), the keys and values for tokens \(1\) through \(n{-}1\) are *exactly the same* as they were at step \(n{-}1\). We're recomputing them for no reason!

The **KV-Cache** stores the key and value projections from all previous tokens, so at each generation step we only need to compute Q, K, V for the *new* token and retrieve cached K, V for the rest.

## Starter Code: Minimal Transformer

Here is a minimal decoder-only transformer. If you completed the ARENA exercises, your implementation may differ in details, but the key ideas are the same. Read through this carefully and make sure you understand each component.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.scale = nn.Parameter(t.ones(d))
        self.eps = eps

    def forward(self, x):
        rms = (x.pow(2).mean(dim=-1, keepdim=True) + self.eps).sqrt()
        return x / rms * self.scale


class Attention(nn.Module):
    def __init__(self, d=64, n_head=4):
        super().__init__()
        self.n_head = n_head
        self.d_head = d // n_head
        self.QKV = nn.Linear(d, 3 * d, bias=False)
        self.O = nn.Linear(d, d, bias=False)

    def forward(self, x):
        b, s, d = x.shape
        q, k, v = self.QKV(x).chunk(3, dim=-1)
        q = q.view(b, s, self.n_head, self.d_head).transpose(1, 2)
        k = k.view(b, s, self.n_head, self.d_head).transpose(1, 2)
        v = v.view(b, s, self.n_head, self.d_head).transpose(1, 2)

        # Causal attention
        attn = (q @ k.transpose(-2, -1)) / (self.d_head ** 0.5)
        mask = t.triu(t.ones(s, s, device=x.device, dtype=t.bool), diagonal=1)
        attn = attn.masked_fill(mask, float('-inf'))
        attn = attn.softmax(dim=-1)
        out = attn @ v

        out = out.transpose(1, 2).reshape(b, s, d)
        return self.O(out)


class MLP(nn.Module):
    def __init__(self, d=64, exp=4):
        super().__init__()
        self.up = nn.Linear(d, exp * d, bias=False)
        self.gate = nn.Linear(d, exp * d, bias=False)
        self.down = nn.Linear(exp * d, d, bias=False)
        self.act = nn.SiLU()

    def forward(self, x):
        return self.down(self.up(x) * self.act(self.gate(x)))


class TransformerBlock(nn.Module):
    def __init__(self, d=64, n_head=4, exp=4):
        super().__init__()
        self.norm1 = RMSNorm(d)
        self.attn = Attention(d, n_head)
        self.norm2 = RMSNorm(d)
        self.mlp = MLP(d, exp)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class Transformer(nn.Module):
    def __init__(self, vocab_size=256, n_ctx=128, d=64, n_head=4, n_layers=4, exp=4):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d)
        self.pos_emb = nn.Embedding(n_ctx, d)
        self.blocks = nn.ModuleList([TransformerBlock(d, n_head, exp) for _ in range(n_layers)])
        self.norm = RMSNorm(d)
        self.unembed = nn.Linear(d, vocab_size, bias=False)

    def forward(self, tokens):
        b, s = tokens.shape
        x = self.tok_emb(tokens) + self.pos_emb(t.arange(s, device=tokens.device))
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        return self.unembed(x)

Verify it works:

In [ ]:
model = Transformer().to(device)
tokens = t.randint(0, 256, (2, 32), device=device)
logits = model(tokens)
print(f"Input shape: {tokens.shape}")
print(f"Output shape: {logits.shape}")  # Should be [2, 32, 256]

---

### Exercise 1 — Implement Naive Autoregressive Generation

Difficulty: 🔴🔴⭕⭕⭕  |  Importance: 🔵🔵🔵⭕⭕  |  ~10 min

Before adding caching, implement naive autoregressive generation. At each step, feed the *entire* sequence through the model and sample the next token from the last position's logits.

In [ ]:
@t.no_grad()
def generate_naive(model, prompt, max_new_tokens=50, temperature=1.0):
    """Generate tokens autoregressively without KV cache.

    Args:
        model: Transformer model
        prompt: tensor of shape (batch, seq_len) with prompt token ids
        max_new_tokens: number of tokens to generate
        temperature: sampling temperature

    Returns:
        tensor of shape (batch, seq_len + max_new_tokens) with generated tokens
    """
    tokens = prompt.clone()
    for _ in range(max_new_tokens):
        # YOUR CODE HERE
        # 1. Forward pass through the model with the full sequence
        # 2. Get logits at the last position
        # 3. Apply temperature and sample
        # 4. Append to the token sequence
        pass
    return tokens

Test it:

In [ ]:
model = Transformer().to(device)
prompt = t.randint(0, 256, (1, 5), device=device)
output = generate_naive(model, prompt, max_new_tokens=20)
assert output.shape == (1, 25), f"Expected shape (1, 25), got {output.shape}"
print("Naive generation works!")

---

### Exercise 2 — Understand the Redundancy

Difficulty: 🔴🔴⭕⭕⭕  |  Importance: 🔵🔵🔵🔵🔵  |  ~10 min

This is a conceptual exercise. Look at the `Attention.forward` method and answer these questions (discuss with your partner or write your answers):


- At generation step \(n\), we compute `q, k, v = self.QKV(x).chunk(3, dim=-1)` for all \(n\) positions. Which of these computations are redundant (i.e., produce the same result as step \(n{-}1\))?

- The causal mask ensures position \(i\) only attends to positions \(\le i\). When generating position \(n\), do the attention outputs at positions \(1, \ldots, n{-}1\) change compared to the previous step?

- If we only need the logit at the *last* position for sampling, what's the minimum amount of computation we need at each step?

---

### Exercise 3 — Implement KVCache

Difficulty: 🔴🔴🔴⭕⭕  |  Importance: 🔵🔵🔵🔵🔵  |  ~15 min

Implement a simple KV-Cache as a Python class. The cache should store key and value tensors for each layer and support appending new K, V entries.

In [ ]:
class KVCache:
    """Stores cached key-value pairs for each layer."""

    def __init__(self, n_layers):
        # YOUR CODE HERE
        # Initialize empty cache for each layer
        # Each layer should store (keys, values) or None
        pass

    def update(self, layer_idx, new_keys, new_values):
        """Append new keys and values to the cache for a given layer.

        Args:
            layer_idx: which transformer layer
            new_keys: tensor of shape (batch, n_head, new_seq, d_head)
            new_values: tensor of shape (batch, n_head, new_seq, d_head)

        Returns:
            (all_keys, all_values): concatenated cached + new tensors
        """
        # YOUR CODE HERE
        pass

Test it:

In [ ]:
cache = KVCache(n_layers=4)
k = t.randn(2, 4, 5, 16)  # batch=2, heads=4, seq=5, d_head=16
v = t.randn(2, 4, 5, 16)
all_k, all_v = cache.update(0, k, v)
assert all_k.shape == (2, 4, 5, 16)

k2 = t.randn(2, 4, 1, 16)  # one new token
v2 = t.randn(2, 4, 1, 16)
all_k, all_v = cache.update(0, k2, v2)
assert all_k.shape == (2, 4, 6, 16), f"Expected seq_len=6 after appending, got {all_k.shape}"
print("KVCache works!")

---

### Exercise 4 — Modify Attention to Use KV-Cache

Difficulty: 🔴🔴🔴🔴⭕  |  Importance: 🔵🔵🔵🔵🔵  |  ~25 min

This is the core exercise. Modify the `Attention` class to optionally accept a KV-Cache. When a cache is provided:


- Compute Q, K, V only for the *input* tokens (which during cached generation is just the new token)

- Concatenate the new K, V with the cached K, V

- Compute attention between the new Q and all (cached + new) K, V

- No causal mask is needed during single-token generation (the new token is the last one and should attend to all previous tokens)

In [ ]:
class CachedAttention(nn.Module):
    def __init__(self, d=64, n_head=4):
        super().__init__()
        self.n_head = n_head
        self.d_head = d // n_head
        self.QKV = nn.Linear(d, 3 * d, bias=False)
        self.O = nn.Linear(d, d, bias=False)

    def forward(self, x, kv_cache=None, layer_idx=None):
        """
        Args:
            x: input tensor (batch, seq_len, d)
            kv_cache: optional KVCache instance
            layer_idx: which layer (required if kv_cache is not None)
        Returns:
            output tensor (batch, seq_len, d)
        """
        b, s, d = x.shape
        q, k, v = self.QKV(x).chunk(3, dim=-1)
        q = q.view(b, s, self.n_head, self.d_head).transpose(1, 2)
        k = k.view(b, s, self.n_head, self.d_head).transpose(1, 2)
        v = v.view(b, s, self.n_head, self.d_head).transpose(1, 2)

        # YOUR CODE HERE
        # If kv_cache is provided:
        #   1. Update the cache with new k, v and get full k, v
        #   2. Compute attention (no causal mask needed for single-token input)
        # If no cache:
        #   Use the standard causal attention as before

        pass

Test it:

In [ ]:
# Test that cached and uncached produce the same output for the prompt
attn = CachedAttention(d=64, n_head=4).to(device)
x = t.randn(1, 10, 64, device=device)

# Uncached (full sequence with causal mask)
out_full = attn(x)

# Cached: first process all tokens to fill the cache, then verify
cache = KVCache(n_layers=1)
out_cached = attn(x, kv_cache=cache, layer_idx=0)

# Both should produce the same output
assert t.allclose(out_full, out_cached, atol=1e-5), "Cached prompt processing should match uncached!"
print("Cached attention matches uncached for prompt processing!")

# Now test single-token forward
x_new = t.randn(1, 1, 64, device=device)
out_new = attn(x_new, kv_cache=cache, layer_idx=0)
assert out_new.shape == (1, 1, 64), f"Expected (1,1,64), got {out_new.shape}"
print("Single-token cached forward works!")

---

### Exercise 5 — Build the Cached Transformer

Difficulty: 🔴🔴🔴⭕⭕  |  Importance: 🔵🔵🔵🔵⭕  |  ~15 min

Now modify the `TransformerBlock` and `Transformer` classes to thread the KV-Cache through. The key changes are:


- `TransformerBlock.forward` takes an optional `kv_cache` and `layer_idx`

- `Transformer.forward` takes an optional `kv_cache` and handles the position embedding correctly (using the position *offset* from the cache)

In [ ]:
class CachedTransformerBlock(nn.Module):
    def __init__(self, d=64, n_head=4, exp=4):
        super().__init__()
        self.norm1 = RMSNorm(d)
        self.attn = CachedAttention(d, n_head)
        self.norm2 = RMSNorm(d)
        self.mlp = MLP(d, exp)

    def forward(self, x, kv_cache=None, layer_idx=None):
        # YOUR CODE HERE
        pass


class CachedTransformer(nn.Module):
    def __init__(self, vocab_size=256, n_ctx=128, d=64, n_head=4, n_layers=4, exp=4):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d)
        self.pos_emb = nn.Embedding(n_ctx, d)
        self.blocks = nn.ModuleList(
            [CachedTransformerBlock(d, n_head, exp) for _ in range(n_layers)]
        )
        self.norm = RMSNorm(d)
        self.unembed = nn.Linear(d, vocab_size, bias=False)
        self.n_layers = n_layers

    def forward(self, tokens, kv_cache=None):
        """
        Args:
            tokens: (batch, seq_len) token ids
            kv_cache: optional KVCache. When provided, tokens are the NEW
                      tokens only, and position embeddings must be offset.
        """
        b, s = tokens.shape

        # YOUR CODE HERE
        # Key: figure out the correct position offset from the cache
        pass

---

### Exercise 6 — Cached Generation

Difficulty: 🔴🔴🔴⭕⭕  |  Importance: 🔵🔵🔵🔵🔵  |  ~15 min

Implement autoregressive generation using the KV-Cache. The generation has two phases:


- **Prefill:** Process the entire prompt at once (filling the cache)

- **Decode:** Generate tokens one at a time, using the cache

In [ ]:
@t.no_grad()
def generate_cached(model, prompt, max_new_tokens=50, temperature=1.0):
    """Generate tokens autoregressively WITH KV cache.

    Args:
        model: CachedTransformer
        prompt: tensor of shape (batch, seq_len)
        max_new_tokens: number of tokens to generate
        temperature: sampling temperature

    Returns:
        tensor of shape (batch, seq_len + max_new_tokens)
    """
    # YOUR CODE HERE
    # 1. Create a KVCache
    # 2. Prefill: forward the prompt through the model (fills the cache)
    # 3. Sample the first new token
    # 4. Decode loop: feed one token at a time, using the cache
    pass

---

### Exercise 7 — Verify Correctness and Benchmark

Difficulty: 🔴🔴⭕⭕⭕  |  Importance: 🔵🔵🔵🔵⭕  |  ~15 min

First, verify that the cached and uncached models produce identical logits for the same input. Then benchmark both approaches.

In [ ]:
# Step 1: Verify correctness
# Create both models with the same weights
model_naive = Transformer(vocab_size=256, n_ctx=128, d=64, n_head=4, n_layers=4).to(device)
model_cached = CachedTransformer(vocab_size=256, n_ctx=128, d=64, n_head=4, n_layers=4).to(device)

# Copy weights from naive to cached
model_cached.load_state_dict(model_naive.state_dict())

prompt = t.randint(0, 256, (1, 10), device=device)

# YOUR CODE HERE
# 1. Run naive model on the prompt, get logits at last position
# 2. Run cached model on the prompt, get logits at last position
# 3. Assert they match (use t.allclose with atol=1e-5)
# 4. Feed one more token through both and check they still match

# Step 2: Benchmark
# YOUR CODE HERE
# Time both generate_naive and generate_cached for max_new_tokens=100
# Print the speedup factor

> **Note:**
**What to expect:** The speedup depends on model size and sequence length. For this small model, you should see a 2–5x speedup. For larger models and longer sequences, KV-caching provides much more dramatic improvements (10–50x or more), since the naive approach recomputes all linear projections for the entire sequence at each step.

## Summary

You've implemented KV-Caching for a decoder-only transformer. The key ideas were:


- During autoregressive generation, K and V for previous tokens never change (due to the causal mask)

- We cache these K, V tensors and only compute new ones for each generated token

- This reduces per-step computation from \(O(n \cdot d)\) to \(O(d)\) for the linear layers

- The position embedding must be offset correctly to account for cached tokens

In [Exercise 4](), we'll revisit KV-caching in the context of frame-autoregressive video models, where it becomes even more important since each "token" is a patch of a video frame.

[Next: Exercise 2 — Flow Matching on MNIST →]()